# 2단계: 데이터 정제 & 구조화
## 흩어진 숫자를 분석 가능한 DataFrame으로

---
### 이 실습에서 배우는 것
- pandas DataFrame 생성 및 조작
- 단위 통일 (백만원 → 억원 → 조원)
- 결측값(NaN) 처리
- 데이터 타입 확인 및 변환

In [ ]:
import pandas as pd
import numpy as np

# 연간 데이터 로드
df = pd.read_csv('data/coway_annual.csv')
print('데이터 크기:', df.shape)
print('\n컬럼 목록:')
print(df.columns.tolist())

## 2-1. 기본 탐색 (항상 먼저 해야 할 것)

In [ ]:
# 데이터 미리보기
df.head()

In [ ]:
# 데이터 타입과 결측값 확인
print('=== 데이터 타입 ===')
print(df.dtypes)
print('\n=== 결측값 개수 ===')
print(df.isnull().sum())

In [ ]:
# 기본 통계 요약
df.describe()

## 2-2. 단위 통일

> IR 자료마다 단위가 다릅니다:
> - 잠정실적 공시: **백만원**
> - Earnings Release: **십억원**
> - IR Book: **억원**
>
> 분석 전에 반드시 하나로 통일!

In [ ]:
# 현재 단위: 백만원
# 억원으로 변환 (÷ 100)
# 조원으로 변환 (÷ 1,000,000)

financial_cols = ['매출액', '영업이익', '법인세전이익', '당기순이익', '국내매출', '해외매출', '기타매출']

df_억 = df.copy()
for col in financial_cols:
    df_억[col] = df[col] / 100  # 백만원 → 억원

print('단위 변환 결과 (억원):')   
print(df_억[['연도', '매출액', '영업이익', '당기순이익']].to_string(index=False))

In [ ]:
# 조원 단위로도 표현해보기
df_조 = df.copy()
for col in financial_cols:
    df_조[col] = df[col] / 1_000_000  # 백만원 → 조원

print('매출액 조원 단위:')
for _, row in df_조.iterrows():
    print(f"{int(row['연도'])}년: {row['매출액']:.2f}조원")

## 2-3. 분기 데이터 정제

In [ ]:
df_q = pd.read_csv('data/coway_quarterly.csv')

# 분기 컬럼을 순서가 있는 인덱스로 설정
df_q = df_q.set_index('분기')

# 억원으로 변환
for col in ['매출액', '영업이익', '법인세전이익', '당기순이익']:
    df_q[col] = df_q[col] / 100

print('분기별 데이터 (억원):')
df_q[['연도', '분기번호', '매출액', '영업이익', '당기순이익']].tail(8)

## 2-4. 파생 컬럼 추가

In [ ]:
df_q_reset = df_q.reset_index()

# 전분기 대비 증감액
df_q_reset['매출_QoQ증감'] = df_q_reset['매출액'].diff()

# 전년 동기 대비 (4분기 차이)
df_q_reset['매출_YoY증감'] = df_q_reset['매출액'].diff(4)

# 분기 레이블 보기 좋게 정리
df_q_reset['분기레이블'] = df_q_reset['연도'].astype(str) + 'Q' + df_q_reset['분기번호'].astype(str)

print(df_q_reset[['분기레이블', '매출액', '매출_QoQ증감', '매출_YoY증감']].tail(8).to_string(index=False))

## 2-5. 해외법인 데이터 구조화

In [ ]:
df_os = pd.read_csv('data/coway_overseas.csv')

# 피벗: 법인별로 열 분리
df_os_pivot = df_os.pivot(index='연도', columns='법인', values='매출액_억원')
df_os_pivot.columns.name = None

print('법인별 연도별 매출액 (억원):')
print(df_os_pivot)

In [ ]:
# 결측값이나 이상값 확인
print('결측값 확인:')
print(df_os_pivot.isnull().sum())

print('\n법인별 통계:')
print(df_os_pivot.describe().round(0))

---
## ✏️ 과제

1. `df_q` 데이터에서 4분기(Q4)만 필터링해서 연도별 4Q 트렌드를 만들어보세요.
   ```python
   df_q4 = df_q[df_q['분기번호'] == 4]
   ```
2. 해외 매출 합계와 `df_annual`의 `해외매출` 컬럼이 일치하는지 검증해보세요.
3. 렌탈 계정수 단위가 '천 계정'인데, '만 계정' 단위 컬럼을 추가해보세요.